In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from listUtils import getFlatList
from master import MasterParams, MusicDBPermDir
from base import MusicDBDir, MusicDBData
from sys import prefix
from pandas import Series, DataFrame, concat
from match import MatchListDataNames, MatchListDataRefs
from musicdb import PanDBIO
from lib.rateyourmusic import RYMUnmatched, RYMLists, RYMParseList, RYMUnknown, fixArtistRefs, getUniqueRefs, removeKnownRefs, fixFloat, getRYMName, getURL, getRYMRefs, getRefData
from numpy import ceil
io = FileIO()
start = True

# Parse Lists

In [ ]:
rmd = RYMParseList(requireID=False)
rmd.parse("/Volumes/Piggy/Charts/data/rymlist6")
rmd.combine()

# Find URLs From Lists To Download

In [ ]:
lrym           = RYMLists(minLists=5, maxLists=70000)
unr            = RYMUnmatched(minRank=1, maxRank=100000)
artistsToMatch = unr.get()
listDataToGet  = lrym.get()

In [ ]:
from match import poolMatchNames
retval = poolMatchNames(baseNames=artistsToMatch["ArtistName"].map(getRYMName).drop_duplicates(), compNames=listDataToGet["Name"].map(getRYMName), nCores=3, progress=True, cutoff=0.95)
results = retval[retval.map(len) > 0]
matchedListArtists = results.apply(lambda x: list(x.index))
matchedListArtists.name = "Ref"
urlsToGet = artistsToMatch.loc[results.index].join(matchedListArtists)
print(f"Found {urlsToGet.shape[0]} Matched Artist Results") 

urefs = getUniqueRefs(urlsToGet)
urlsToGet = urlsToGet.join(urefs)
nAllRefs = urlsToGet["Ref"].map(len).sum()
nAllURefs = urlsToGet["URefs"].map(len).sum()
print(f"Found {nAllURefs}/{nAllRefs} Refs To Download")
io.save(idata=urlsToGet, ifile="urlsToGet.p")

In [ ]:
#https://rateyourmusic.com/artist/anonymous_f8
urlsToGet = io.get("urlsToGet.p")
urlsToGet = urlsToGet.sort_values(by="Rank", ascending=True)
rymRefs   = getRYMRefs()
print(f"# Found {len(rymRefs)} Known RateYourMusic Refs")
print(f"# Found {urlsToGet.shape[0]} Artists URLs To Get")
masterRefs = {} if start is True else masterRefs
urlsToGet["ToGet"] = urlsToGet["URefs"].apply(lambda x: removeKnownRefs(x, rymRefs, masterRefs))
done = urlsToGet[urlsToGet["ToGet"].map(len) == 0]
print(f"# Found {done.shape[0]} Artists With Known URLs")
refsToGet = urlsToGet[(urlsToGet["ToGet"].map(len) > 0)]
print(f"# Found {refsToGet.shape[0]} Artists URLs To Download")

In [ ]:
io.save(idata=refsToGet, ifile="refsToGet.p")

In [ ]:
head = 7
hset = 12
N    = refsToGet.shape[0]
nT   = int(ceil(N/hset))
for i,(_,row) in enumerate(refsToGet[((head-1)*hset):((head)*hset)].iterrows()):
    refs   = row["URefs"]
    name   = row["ArtistName"]
    rank   = fixFloat(row["Rank"])
    counts = fixFloat(row["Counts"])
    n      = (head-1)*hset+i+1
    for ref in refs:
        url    = "https://rateyourmusic.com{0}".format(ref)
        print(f"{head: >3} / {nT: <3} | {n: >5} / {N: <4} |  {name: <40}{rank: <10}{counts: <4} | {url}")

# Find URLs From Relationships To Download

In [ ]:
#############################################################################################################################################################################################
#############################################################################################################################################################################################
mio      = rateyourmusic.MusicDBIO(verbose=False)
refData  = mio.data.getSummaryRefData()
pdbio    = PanDBIO()
mmeDF    = pdbio.getData()
mmeDF["Rank"] = mmeDF["PrimaryRank"].apply(lambda rank: rank[0])
rymID    = mmeDF[mmeDF["RateYourMusic"].notna()]["RateYourMusic"]
unknown  = mmeDF[mmeDF["RateYourMusic"].isna()][["ArtistName", "Rank", "Counts"]]

#############################################################################################################################################################################################
#############################################################################################################################################################################################
minCounts = None
maxCounts = None
minRank   = 30000
maxRank   = 31000

#############################################################################################################################################################################################
#############################################################################################################################################################################################
print("="*200)
artistsToMatch = unknown
print("Found {0: >6} Unmatched Master Artsts".format(artistsToMatch.shape[0]))
if isinstance(minCounts,int):
    artistsToMatch = artistsToMatch[(artistsToMatch["Counts"] > minCounts)]
    print("Found {0: >6} Unmatched Master Artsts With Counts > {1}".format(artistsToMatch.shape[0], minCounts))
if isinstance(maxCounts,int):
    artistsToMatch = artistsToMatch[(artistsToMatch["Counts"] <= maxCounts)]
    print("Found {0: >6} Unmatched Master Artsts With Counts <= {1}".format(artistsToMatch.shape[0], maxCounts))
if isinstance(minRank,int):
    artistsToMatch = artistsToMatch[(artistsToMatch["Rank"] > minRank)]
    print("Found {0: >6} Unmatched Master Artsts With Rank > {1}".format(artistsToMatch.shape[0], minRank))
if isinstance(maxRank,int):
    artistsToMatch = artistsToMatch[(artistsToMatch["Rank"] <= maxRank)]
    print("Found {0: >6} Unmatched Master Artsts With Rank <= {1}".format(artistsToMatch.shape[0], maxRank))
print("="*200)

In [ ]:
from lib.rateyourmusic import Relationships
rts = Relationships()
mem = rts.getMem()
mof = rts.getMof()
asa = rts.getAsa()
rar = rts.getRar()
refdf = concat([fixArtistRefs(mem["Refs"]), fixArtistRefs(mof["Refs"]), fixArtistRefs(rar["Refs"]), fixArtistRefs(asa["Refs"])]).drop_duplicates()
print("Found {0: >6} Total Relationship Artists".format(refdf.shape[0]))

In [ ]:
from match import poolMatchNames
retval = poolMatchNames(baseNames=artistsToMatch["ArtistName"].map(getRYMName).drop_duplicates(), compNames=refdf["Name"].map(getRYMName), nCores=3, progress=True, cutoff=0.95)
results = retval[retval.map(len) > 0]
matchedRelationArtists = results.apply(lambda x: list(x.index))
matchedRelationArtists.name = "Ref"
urlsToGet = artistsToMatch.loc[results.index].join(matchedRelationArtists)
print(f"Found {urlsToGet.shape[0]} Matched Artist Results") 

urefs = getUniqueRefs(urlsToGet)
urlsToGet = urlsToGet.join(urefs)
nAllRefs = urlsToGet["Ref"].map(len).sum()
nAllURefs = urlsToGet["URefs"].map(len).sum()
print(f"Found {nAllURefs}/{nAllRefs} Refs To Download")
io.save(idata=urlsToGet, ifile="urlsToGet25000.p")

# Download Found URLs

In [ ]:
urlsToGet = io.get("urlsToGet.p")
urlsToGet = urlsToGet.sort_values(by="Rank", ascending=True)
rymRefs   = getRYMRefs()
print(f"# Found {len(rymRefs)} Known RateYourMusic Refs")
print(f"# Found {urlsToGet.shape[0]} Artists URLs To Get")
masterRefs = {}
urlsToGet["ToGet"] = urlsToGet["URefs"].apply(lambda x: removeKnownRefs(x, rymRefs, masterRefs))
done = urlsToGet[urlsToGet["ToGet"].map(len) == 0]
print(f"# Found {done.shape[0]} Artists With Known URLs")
refsToGet = urlsToGet[(urlsToGet["ToGet"].map(len) > 0)]
print(f"# Found {refsToGet.shape[0]} Artists URLs To Download")

In [ ]:
head = 3
hset = 12
N    = refsToGet.shape[0]
nT   = int(ceil(N/hset))
for i,(_,row) in enumerate(refsToGet[((head-1)*hset):((head)*hset)].iterrows()):
    refs   = row["URefs"]
    name   = row["ArtistName"]
    rank   = fixFloat(row["Rank"])
    counts = fixFloat(row["Counts"])
    n      = (head-1)*hset+i+1
    for ref in refs:
        url    = "https://rateyourmusic.com{0}".format(ref)
        print(f"{head: >3} / {nT: <3} | {n: >5} / {N: <4} |  {name: <40}{rank: <10}{counts: <4} | {url}")

# Check Unknown (For New List)

In [ ]:
if False:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    pdbio.addMetrics()
    pdbio.setIndex()

In [ ]:
rymunk  = RYMUnknown()
rymID   = rymunk.rymID
unknown = rymunk.unknown

In [ ]:
unknown.head(30)

In [ ]:
unknown[(unknown["Rank"] > 22000) & (unknown["Rank"] <= 22500)]

# Parse

In [ ]:
from lib import rateyourmusic
rateyourmusic.moveLocalFiles()
rateyourmusic.removeLocalFiles()

from utils import PoolIO
pio = PoolIO("RateYourMusic")
pio.parse()
pio.meta()
pio.sum()
pio.search()

from lib.rateyourmusic import Relationships
rts = Relationships()
rts.compute()

if False:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    pdbio.addMetrics()
    pdbio.setIndex()